In [1]:
import sys
sys.path.append("../src")

import os
from typing import Dict, Any

import torch
import numpy as np
import pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from utils import *

In [2]:
gene_expression_path = os.path.join("..", "data", "raw", "gene-expression-normalized.csv")

df = pd.read_csv(gene_expression_path, index_col=0)

df.head()

,A1BG,A1CF,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AADAC,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
ACH-000828,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ACH-000568,0.0,0.122203,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ACH-000560,0.0,0.152391,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ACH-000561,0.0,0.160657,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ACH-000562,0.0,0.161598,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
model_path = os.path.join("..", "assets", "models", "models.ckpt")

os.path.isfile(model_path)

True

In [4]:
model, config = load_model_frommmf(model_path)

/Users/ericmonzon/Desktop/personal projects/scFoundation-CDR/notebooks/../src/utils/load.py:126: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_data = torch.load(best_c

{'mask_gene_name': False, 'gene_num': 19266, 'seq_len': 19266, 'encoder': {'hidden_dim': 768, 'depth': 12, 'heads': 12, 'dim_head': 64, 'seq_len': 19266, 'module_type': 'transformer', 'norm_first': False}, 'decoder': {'hidden_dim': 512, 'depth': 6, 'heads': 8, 'dim_head': 64, 'module_type': 'performer', 'seq_len': 19266, 'norm_first': False}, 'n_class': 104, 'pad_token_id': 103, 'mask_token_id': 102, 'bin_num': 100, 'bin_alpha': 1.0, 'rawcount': True, 'model': 'mae_autobin', 'test_valid_train_idx_dict': '/nfs_beijing/minsheng/data/os10000w-new/global_shuffle/meta.csv.train_set_idx_dict.pt', 'valid_data_path': '/nfs_beijing/minsheng/data/valid_count_10w.npz', 'num_tokens': 13, 'train_data_path': None, 'isPanA': False, 'isPlanA1': False, 'max_files_to_load': 5, 'bin_type': 'auto_bin', 'value_mask_prob': 0.3, 'zero_mask_prob': 0.03, 'replace_prob': 0.8, 'random_token_prob': 0.1, 'mask_ignore_token_ids': [0], 'decoder_add_zero': True, 'mae_encoder_max_seq_len': 15000, 'isPlanA': False, 'ma

In [5]:
drug_dir = os.path.join("..", "data", "cleaned", "drugs")
table_path = os.path.join("..", "data", "cleaned", "train.csv")

In [6]:
dataset = MultiModalDataset(
    drug_dir=drug_dir,
    table_path=table_path,
    gene_expression_path=gene_expression_path,
    pretrain_config=config
)

loader = DataLoader(dataset, batch_size=5)

In [7]:
drug_dict, expression_dict, target = next(iter(loader))

x, x_padding, position_gene_ids = process_expression_dict(expression_dict)

In [8]:
drug_dict

{'feature_path': ['../data/cleaned/drugs/drug-feature.npy',
  '../data/cleaned/drugs/drug-feature.npy',
  '../data/cleaned/drugs/drug-feature.npy',
  '../data/cleaned/drugs/drug-feature.npy',
  '../data/cleaned/drugs/drug-feature.npy'],
 'edge_list_path': ['../data/cleaned/drugs/11282283/drug-edge-list.npy',
  '../data/cleaned/drugs/216326/drug-edge-list.npy',
  '../data/cleaned/drugs/6918289/drug-edge-list.npy',
  '../data/cleaned/drugs/56965967/drug-edge-list.npy',
  '../data/cleaned/drugs/300471/drug-edge-list.npy']}

In [9]:
# x = expression_dict["x"]
# position_gene_ids = expression_dict["position_gene_ids"]
# x_padding = expression_dict["x_padding"]

In [10]:
type(model)

utils.pretrainmodels.mae_autobin.MaeAutobin

In [11]:
encoder = scFoundationEncoder(model)

encoder

scFoundationEncoder(
  (token_embedder): AutoDiscretizationEmbedding2(
    (mlp): Linear(in_features=1, out_features=100, bias=True)
    (mlp2): Linear(in_features=100, out_features=100, bias=True)
    (LeakyReLU): LeakyReLU(negative_slope=0.1)
    (Softmax): Softmax(dim=-1)
    (emb): Embedding(100, 768)
    (emb_mask): Embedding(1, 768)
    (emb_pad): Embedding(1, 768)
  )
  (position_embedder): Embedding(19267, 768)
  (encoder): pytorchTransformerModule(
    (transformer_encoder): ModuleList(
      (0-11): 12 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (linear1): Linear(in_features=768, out_features=3072, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=3072, out_features=768, bias=True)
        (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((768,), eps

In [12]:
gene_embedding = encoder(x, x_padding, position_gene_ids)

In [16]:
x_padding

tensor([[False, False, False,  ...,  True,  True,  True],
        [False, False, False,  ...,  True,  True,  True],
        [False, False, False,  ..., False, False, False],
        [False, False, False,  ...,  True,  True,  True],
        [False, False, False,  ...,  True,  True,  True]])

In [17]:
position_gene_ids

tensor([[    1,    96,   101,  ...,   103,   103,   103],
        [    1,    96,   101,  ...,   103,   103,   103],
        [    1,    96,   101,  ..., 19221, 19264, 19265],
        [    1,    96,   101,  ...,   103,   103,   103],
        [   96,   101,   102,  ...,   103,   103,   103]])

In [18]:
x

tensor([[5.8631e-02, 3.0388e+00, 3.0734e+00,  ..., 1.0300e+02, 1.0300e+02,
         1.0300e+02],
        [2.7168e+00, 2.9122e+00, 2.9463e+00,  ..., 1.0300e+02, 1.0300e+02,
         1.0300e+02],
        [1.3358e+00, 2.8290e+00, 2.8622e+00,  ..., 3.0211e+00, 1.6404e+03,
         1.6404e+03],
        [1.5537e-01, 2.8587e+00, 3.1463e+00,  ..., 1.0300e+02, 1.0300e+02,
         1.0300e+02],
        [3.0745e+00, 2.9809e+00, 3.0703e+00,  ..., 1.0300e+02, 1.0300e+02,
         1.0300e+02]], dtype=torch.float64)

In [13]:
gene_embedding.shape

torch.Size([5, 768])

In [14]:
x_padding

tensor([[False, False, False,  ...,  True,  True,  True],
        [False, False, False,  ...,  True,  True,  True],
        [False, False, False,  ..., False, False, False],
        [False, False, False,  ...,  True,  True,  True],
        [False, False, False,  ...,  True,  True,  True]])

In [15]:
target

tensor([0, 0, 0, 1, 1])